# Marta Negri 
# Luca Davì 

## Taks 2: 
assessing the performance of pre-trained models in a zero-shot setting, where no additional training on waste images is performed. This will provide insights
into the generalization capabilities of modern detectors such as YOLO, DETR, and RT-DETR.
After the initial evaluation, fine-tune a small variant of the RT-DETR model on the training split
of the waste images dataset, and assess and discuss its performance, comparing it to the other
models

### 2.1 Dataset Preparation

In [1]:
%pip install pycocotools

In [2]:
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import torchvision
import torch
import matplotlib.pyplot as plt
import numpy as np


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print(f'Using device: {device}')

Using device: cuda


In [4]:

train_dataset = torchvision.datasets.CocoDetection(root='data/TACO_subset', annFile='data/TACO_subset/annotations_train.json', transform=None)
test_dataset = torchvision.datasets.CocoDetection(root='data/TACO_subset', annFile='data/TACO_subset/annotations_test.json', transform=None)
val_dataset = torchvision.datasets.CocoDetection(root='data/TACO_subset', annFile='data/TACO_subset/annotations_val.json', transform=None)
#yolo and detr require images in different formats with different pipelines

train_dataset[0]

loading annotations into memory...


FileNotFoundError: [Errno 2] No such file or directory: 'data/TACO_subset/annotations_train.json'

In [ ]:
train_length = len(train_dataset)
test_length = len(test_dataset)
val_length = len(val_dataset)
tot_length = train_length + test_length + val_length

print(f"Train dataset length: {train_length}: {train_length/tot_length*100:.2f}% of total")
print(f"Test dataset length: {test_length}: {test_length/tot_length*100:.2f}% of total")
print(f"Validation dataset length: {val_length}: {val_length/tot_length*100:.2f}% of total")

In [ ]:
# Visualize three samples from each split with the corresponding category labels with category names

indexes = torch.randint(0, min(train_length, test_length, val_length), (3,)).tolist()

fig, axs = plt.subplots(3, 3, figsize=(12, 12))
for i in range(3):
    axs[0, i].imshow(train_dataset[indexes[i]][0])
    axs[0, i].set_title(f"Train Sample {i+1}\nCategory: {train_dataset[indexes[i]][1][0]['category_id']}\nName: {train_dataset.coco.loadCats(train_dataset[indexes[i]][1][0]['category_id'])[0]['name']}")
    axs[0, i].axis('off')
    
    axs[1, i].imshow(test_dataset[indexes[i]][0])
    axs[1, i].set_title(f"Test Sample {i+1}\nCategory: {test_dataset[indexes[i]][1][0]['category_id']}\nName: {test_dataset.coco.loadCats(test_dataset[indexes[i]][1][0]['category_id'])[0]['name']}")
    axs[1, i].axis('off')
    
    axs[2, i].imshow(val_dataset[indexes[i]][0])
    axs[2, i].set_title(f"Validation Sample {i+1}\nCategory: {val_dataset[indexes[i]][1][0]['category_id']}\nName: {val_dataset.coco.loadCats(val_dataset[indexes[i]][1][0]['category_id'])[0]['name']}")
    axs[2, i].axis('off')

subtask 2.2. zero-shot detection

In [ ]:
#define dataloaders for each split
def collate_fn(batch): #standard collate function for object detection tasks provided in pytorch documentation
    return tuple(zip(*batch))
BATCH_SIZE = 4
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
#load models
from ultralytics import YOLO
from transformers import pipeline

yolo_model = YOLO('yolo11m.pt').to(device)
detr_model = pipeline('object-detection', model='facebook/detr-resnet-50',device=device)
rt_detr_model = pipeline('object-detection', model='PekingU/rtdetr_r50vd',device=device)

print(f'Models are on device: \nYOLO: {yolo_model.device}\nDETR device: {detr_model.device}\nRT-DETR device: {rt_detr_model.device}')


In [ ]:
def dataset_to_torchmetrics(target):
    #convert coco annotations to torchmetrics format
    boxes = []
    labels = []
    for ann in target:
        box = ann['bbox']
        #coco format is [x_min, y_min, width, height], convert to [x_min, y_min, x_max, y_max]
        box = [box[0], box[1], box[0] + box[2], box[1] + box[3]]
        boxes.append(box)
        labels.append(0) #we don't consider classes, set all to 0
    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.int64)}

def yolo_to_torchmetrics(yolo_results, original_size, IMG_SIZE=640):
    yolo_boxes = yolo_results.boxes.xyxy
    # Rescale boxes to original image size
    yolo_boxes[:, [0, 2]] *= original_size[0] / IMG_SIZE
    yolo_boxes[:, [1, 3]] *= original_size[1] / IMG_SIZE
    return {
        "boxes": yolo_boxes,
        "scores": yolo_results.boxes.conf,
        "labels": torch.zeros(len(yolo_results.boxes.xyxy), dtype=torch.int64), # Ignoring class
    }

def detr_to_torchmetrics(detr_results, confidence_threshold=0.1):
    detr_results = [res for res in detr_results if res['score'] >= confidence_threshold]
    if len(detr_results) == 0:
        return {
            "boxes": torch.empty((0, 4), dtype=torch.float32),
            "scores": torch.empty((0,), dtype=torch.float32),
            "labels": torch.empty((0,), dtype=torch.int64), # Ignoring class
        }
    boxes = []
    scores = []
    for res in detr_results:
        box = res['box']
        if isinstance(box, dict):
            boxes.append([box['xmin'], box['ymin'], box['xmax'], box['ymax']])
        else:
            x,y,w,h = box
            boxes.append([x, y, x + w, y + h])
        scores.append(res['score'])
    return {
        "boxes": torch.tensor(boxes, dtype=torch.float32),
        "scores": torch.tensor(scores, dtype=torch.float32),
        "labels": torch.zeros(len(boxes), dtype=torch.int64), # Ignoring class
    }

In [ ]:
def yolo_top_IoU(yolo_pred, target, n_boxes=10):
    # receive boxes predictions and target boxes for single image
    # select top-10 predictions based on confidence
    # for each target box compute IoU with 10 predictions
    # select prediction with highest IoU [delete from set]
    # final set of predictions is given to mAP
    top_indexes = torch.argsort(yolo_pred["scores"], descending=True)[:n_boxes]
    selected_boxes = yolo_pred["boxes"][top_indexes]
    selected_scores = yolo_pred["scores"][top_indexes]

    #ensure same device
        # selected_boxes = selected_boxes.to(device)
        # selected_scores = selected_scores.to(device)
        # target_boxes = target['boxes'].to(device)

    ious = torchvision.ops.box_iou(selected_boxes, target["boxes"])  # shape: (n_selected, n_target)
    matched_indexes = set()
    final_boxes = []
    final_scores = []

    for target_idx in range(ious.size(1)):
        available_indexes = [i for i in range(ious.size(0)) if i not in matched_indexes]
        if not available_indexes:
            break
        target_ious = ious[available_indexes, target_idx]
        best_iou_idx = available_indexes[torch.argmax(target_ious).item()]
        matched_indexes.add(best_iou_idx)
        final_boxes.append(selected_boxes[best_iou_idx].unsqueeze(0))
        final_scores.append(selected_scores[best_iou_idx].unsqueeze(0))
    
    return {
        "boxes": torch.cat(final_boxes, dim=0) if final_boxes else torch.empty((0,4), dtype=torch.float32),
        "scores": torch.cat(final_scores, dim=0) if final_scores else torch.empty((0,), dtype=torch.float32),
        "labels": torch.zeros(len(final_boxes), dtype=torch.int64), # Ignoring class
    }

def detr_top_IoU(detr_pred, target, n_boxes=10):
    # receive boxes predictions and target boxes for single image
    # select top-10 predictions based on confidence
    # for each target box compute IoU with 10 predictions
    # select prediction with highest IoU [delete from set]
    # final set of predictions is given to mAP

    top_indexes = torch.argsort(detr_pred['scores'], descending=True)[:n_boxes]
    selected_boxes = detr_pred['boxes'][top_indexes]
    selected_scores = detr_pred['scores'][top_indexes]

    ious = torchvision.ops.box_iou(selected_boxes, target['boxes'])  # shape: (n_selected, n_target)
    matched_indexes = set()
    final_boxes = []
    final_scores = []

    for target_idx in range(ious.size(1)):
        available_indexes = [i for i in range(ious.size(0)) if i not in matched_indexes]
        if not available_indexes:
            break
        target_ious = ious[available_indexes, target_idx]
        best_iou_idx = available_indexes[torch.argmax(target_ious).item()]
        matched_indexes.add(best_iou_idx)
        final_boxes.append(selected_boxes[best_iou_idx].unsqueeze(0))
        final_scores.append(selected_scores[best_iou_idx].unsqueeze(0))
    
    return {
        "boxes": torch.cat(final_boxes, dim=0) if final_boxes else torch.empty((0,4), dtype=torch.float32),
        "scores": torch.cat(final_scores, dim=0) if final_scores else torch.empty((0,), dtype=torch.float32),
        "labels": torch.zeros(len(final_boxes), dtype=torch.int64), # Ignoring class
    }

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
IMG_SIZE = 640
def evaluate_models(yolo_model, detr_model,rt_detr_model, data_loader):

    metric_yolo = MeanAveragePrecision(class_metrics=False)
    metric_detr = MeanAveragePrecision(class_metrics=False)
    metric_rt_detr = MeanAveragePrecision(class_metrics=False)

    iou_metric_yolo = MeanAveragePrecision(class_metrics=False)
    iou_metric_detr = MeanAveragePrecision(class_metrics=False)
    iou_metric_rt_detr = MeanAveragePrecision(class_metrics=False)
    # yolo_confs = []
    # detr_scores = []
    # rt_detr_scores = []
    with torch.no_grad():
        for images, targets in data_loader:
            original_sizes = [img.size for img in images]  # Store original sizes for image rescaling
            yolo_batch = torch.stack(
                [torchvision.transforms.ToTensor()(
                    torchvision.transforms.Resize((IMG_SIZE, IMG_SIZE))(img))
                    for img in images]) #YOLO is happier with tensor [B,C,W,H], but doesn't handle resizing internally

            #YOLO inference
            yolo_results = yolo_model(yolo_batch, verbose=False)
            #DETR inference
            detr_batch = [img.convert("RGB") if img.mode != "RGB" else img for img in images] #DETR needs PIL RGB images
            detr_results = detr_model(detr_batch)
            #RT-DETR inference
            rt_detr_results = rt_detr_model(detr_batch)
            
            #transform target to map format
            targets = [dataset_to_torchmetrics(t) for t in targets]
            #yolo_results are the only one on device instead of cpu, move them
            yolo_results = [res.to('cpu') for res in yolo_results]
            #targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            #determine top k predictions for each model, where k is the number of ground truth boxes in the image
            for og_size,target,yolo_res,detr_res,rt_detr_res in zip(original_sizes,targets,yolo_results,detr_results,rt_detr_results):
                k = len(target)
                yolo_sorted_indices = torch.argsort(yolo_res.boxes.conf, descending=True)[:k]
                yolo_topk = yolo_res[yolo_sorted_indices]

                detr_topk = sorted(detr_res, key=lambda x: x['score'], reverse=True)[:k]
                rt_detr_topk = sorted(rt_detr_res, key=lambda x: x['score'], reverse=True)[:k]

                metric_yolo.update([yolo_to_torchmetrics(yolo_topk, og_size, IMG_SIZE)], [target])
                metric_detr.update([detr_to_torchmetrics(detr_topk)], [target])
                metric_rt_detr.update([detr_to_torchmetrics(rt_detr_topk)], [target])

                # print(target['boxes'].device)
                # print(yolo_res.boxes.xyxy.device)
                # print(detr_to_torchmetrics(detr_res)['boxes'].device)

                iou_metric_yolo.update([yolo_top_IoU(yolo_to_torchmetrics(yolo_res, og_size, IMG_SIZE), target, n_boxes=10)], [target])
                iou_metric_detr.update([detr_top_IoU(detr_to_torchmetrics(detr_res), target, n_boxes=10)], [target])
                iou_metric_rt_detr.update([detr_top_IoU(detr_to_torchmetrics(rt_detr_res), target, n_boxes=10)], [target])
                
    return metric_yolo.compute(), metric_detr.compute(), metric_rt_detr.compute(), iou_metric_yolo.compute(), iou_metric_detr.compute(), iou_metric_rt_detr.compute()           


In [ ]:
print("Evaluating models on the test dataset...")

print(len(test_loader))
yolo_map, detr_map, rt_detr_map, iou_yolo_map, iou_detr_map, iou_rt_detr_map = evaluate_models(yolo_model, detr_model, rt_detr_model, test_loader)
print(f"YOLO mAP: {yolo_map}")
print(f"DETR mAP: {detr_map}")
print(f"RT-DETR mAP: {rt_detr_map}")

print(f"YOLO IoU-based mAP: {iou_yolo_map}")
print(f"DETR IoU-based mAP: {iou_detr_map}")
print(f"RT-DETR IoU-based mAP: {iou_rt_detr_map}")

## Task 2.3. finetuning detr

In [ ]:
from transformers import AutoImageProcessor
IMG_SIZE = 480
checkpoint = "PekingU/rtdetr_v2_r50vd"
image_processor = AutoImageProcessor.from_pretrained(
    checkpoint,
    do_resize=True,
    size={"height": IMG_SIZE, "width": IMG_SIZE},
    use_fast=True,
)

label2id = {'Plastic bag & wrapper': 0,
  'Cigarette': 1,
  'Bottle': 2,
  'Bottle cap': 3,
  'Can': 4,
  'Carton': 5}

id2label = {v: k for k, v in label2id.items()}
label2id, id2label
categories = list(id2label.values())

train_augmentation_and_transform = ...
validation_transform = ...


In [ ]:
#load datasets with provided custom dataset class

from taco_dataset import TACODETRDetectionDataset

train_dataset = TACODETRDetectionDataset(img_folder='data/TACO_subset', ann_file='data/TACO_subset/annotations_train.json',processor=image_processor)
test_dataset = TACODETRDetectionDataset(img_folder='data/TACO_subset', ann_file='data/TACO_subset/annotations_test.json',processor=image_processor)
val_dataset = TACODETRDetectionDataset(img_folder='data/TACO_subset', ann_file='data/TACO_subset/annotations_val.json',processor=image_processor)

In [ ]:
from PIL import Image, ImageDraw
for i in [15, 16, 17]:
    sample = train_dataset[i]

    # De-normalize image
    image = sample["pixel_values"]
    print("Image tensor shape:", image.shape)
    image = image.numpy().transpose(1, 2, 0)
    image = (image - image.min()) / (image.max() - image.min()) * 255.
    image = Image.fromarray(image.astype(np.uint8))

    # Convert boxes from [center_x, center_y, width, height] to [x, y, width, height] for visualization
    boxes = sample["labels"]["boxes"].numpy()
    print("Boxes shape:", boxes.shape)
    boxes[:, :2] = boxes[:, :2] - boxes[:, 2:] / 2
    w, h = image.size
    boxes = boxes * np.array([w, h, w, h])[None]

    categories = sample["labels"]["class_labels"].numpy()
    print("Categories shape:", categories.shape)

    # Draw boxes and labels on image
    draw = ImageDraw.Draw(image)
    for box, category in zip(boxes, categories):
        x, y, w, h = box
        draw.rectangle([x, y, x + w, y + h], outline="red", width=1)
        draw.text((x, y), id2label[category], fill="white")

    display(image)

In [ ]:
from dataclasses import dataclass
from transformers.image_transforms import center_to_corners_format

@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor

class MAPEvaluator:

    def __init__(self, image_processor, threshold=0.00, id2label=None):
        self.image_processor = image_processor
        self.threshold = threshold
        self.id2label = id2label
    def collect_image_sizes(self,targets):
        image_sizes = []
        for batch in targets:
            batch_image_sizes = torch.tensor(np.array([x['size'] for x in batch]))
            image_sizes.append(batch_image_sizes)
        return image_sizes
    def collect_targets(self, targets, image_sizes):
        post_processed_targets=[]
        for target_batch, image_size_batch in zip(targets, image_sizes):
            batch_targets = []
            for target, size in zip(target_batch, image_size_batch):
                height, width = size
                boxes = torch.tensor(target['boxes'])
                boxes = center_to_corners_format(boxes)
                boxes = boxes * torch.tensor([width, height, width, height], dtype=torch.float32)

                labels = torch.tensor(target['labels'])
                post_processed_targets.append({'boxes': boxes, 'labels': labels})
        return post_processed_targets
    
    def collect_predictions(self, predictions, image_sizes):
        post_processed_predictions = []
        for batch, target_sizes in zip(predictions, image_sizes):
            batch_logits, batch_boxes = batch[1], batch[2]
            output = ModelOutput(logits=torch.tensor(batch_logits), pred_boxes=torch.tensor(batch_boxes))
            post_processed_output = self.image_processor.post_process_object_detection(
                output, threshold=self.threshold, target_sizes=target_sizes
            )
            post_processed_predictions.extend(post_processed_output)
        return post_processed_predictions
    
    @torch.no_grad()
    def __call__(self, evaluation_results):

        predictions, targets = evaluation_results.predictions, evaluation_results.label_ids

        image_sizes = self.collect_image_sizes(targets)
        post_processed_targets = self.collect_targets(targets, image_sizes)
        post_processed_predictions = self.collect_predictions(predictions, image_sizes)

        evaluator = MeanAveragePrecision(box_format='xyxy', class_metrics=True)
        evaluator.warn_on_many_detections = False
        evaluator.update(post_processed_predictions, post_processed_targets)

        metrics = evaluator.compute()

        classes = metrics.pop("classes")
        map_per_class = metrics.pop("map_per_class")
        mar_100_per_class = metrics.pop("mar_100_per_class")

        for class_id, class_map, class_mar in zip(classes, map_per_class, mar_100_per_class):
            class_name = self.id2label[class_id.item()] if self.id2label is not None else class_id.item()
            metrics[f"map_{class_name}"] = class_map
            metrics[f"mar_100_{class_name}"] = class_mar
        
        metrics = {k: round(v.item(), 4) for k, v in metrics.items()}
        return metrics
    

eval_compute_metrics_fn = MAPEvaluator(image_processor=image_processor, threshold=0.1, id2label=None)

In [ ]:
from transformers import AutoModelForObjectDetection

model = AutoModelForObjectDetection.from_pretrained(
    checkpoint,
    id2label= id2label,
    label2id= label2id,
    ignore_mismatched_sizes=True,
)

In [ ]:
from taco_dataset import taco_detr_collate_fn
collate_fn = taco_detr_collate_fn

In [ ]:

os.environ['WANDB_MODE']= 'offline'

from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir=f"{checkpoint}-finetuned-taco",
    num_train_epochs=1,
    max_grad_norm=0.1,
    learning_rate=5e-5,
    warmup_steps=300,
    per_device_train_batch_size=8,
    dataloader_num_workers=2,
    metric_for_best_model="eval_map",
    greater_is_better=True,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,
    remove_unused_columns=False,
    eval_do_concat_batches=False,
    report_to="wandb",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=image_processor,
    data_collator = collate_fn,
    compute_metrics = eval_compute_metrics_fn,
)

#trainer.train()

In [ ]:
trainer.train()

In [ ]:
print(torch.__version__)
print(torchvision.__version__)

In [ ]:
! pip install 'accelerate>=0.26.0'